# WAF Assessment — Configuration

**Run this first, or `%run` it from a pillar notebook.** Every pillar notebook and the
driver notebook load their settings from here, so this is the single place to configure
the assessment.

| Setting | What it does |
| --- | --- |
| `CATALOG` / `SCHEMA` | Where results are persisted so pillars can be combined |
| `CLOUD_PROVIDER` | `azure` (default), `aws` or `gcp` - picks the GPU, spot and instance-generation rules |
| `PROD_CATALOG_PATTERNS` | Which catalogs count as production |
| `LOOKBACK_DAYS` | Activity window for telemetry-based checks |
| `COMPLETE_AT` | Coverage needed to score `completed` (default 80%) |
| `INVENTORY_WINDOW_DAYS` | Audit-log window used to rebuild account-wide inventories (default 365) |

**Run as an account admin** (also a workspace admin here). One run covers every workspace
in this metastore's region: per-workspace objects (cluster policies, Git folders, secret
scopes, Apps, warehouses, jobs) are read from system tables and the audit log, and
account identity counts from the workspace's account-SCIM endpoint. The assessment only
reads your environment; its only writes go to the `CATALOG`.`SCHEMA` you set below.

Fill in the **REQUIRED** values below before running anything.

## 1. Required settings

`CATALOG` and `SCHEMA` are where the assessment writes its own results table. Pick a
location you can write to — the assessment never writes anywhere else.

In [0]:
# ============================================================================
# REQUIRED - fill these in
# ============================================================================

# Catalog and schema for the assessment's own results table.
# The schema is created if missing; the catalog must already exist unless you
# set CREATE_CATALOG = True below.
CATALOG = ""   # e.g. "main"
SCHEMA = ""    # e.g. "waf_assessment"

# Cloud this account runs on: "azure" (default), "aws" or "gcp". Instance names, GPU
# families and spot settings differ per cloud, so CO-01-05 (GPU), CO-01-07 (instance
# generation) and CO-04-02 (spot) read the rules for this cloud.
CLOUD_PROVIDER = "azure"

# ============================================================================
# Production scope - review these
# ============================================================================

# Catalogs matching any of these SQL LIKE patterns are treated as production.
# Governance questions are scored against production because that is where it
# matters. If NOTHING matches, the notebooks fall back to all non-system
# catalogs and stamp a caveat on every affected rationale.
PROD_CATALOG_PATTERNS = ["prod%", "main", "%_prod", "%_production"]

# Schemas matching these patterns are excluded even inside a production catalog.
PROD_SCHEMA_EXCLUDE_PATTERNS = ["%dev%", "%test%", "%sandbox%", "%tmp%", "%scratch%"]

# Never assessed as your Databricks environment's data.
SYSTEM_CATALOGS = ["system", "__databricks_internal%", "samples"]

# ============================================================================
# Cost - runtime currency & chargeback (CO-01-04, CO-03-03)
# ============================================================================

# CO-01-04: clusters are "up to date" if within a couple major DBR versions of the
# newest one in your fleet. None = auto (self-calibrating); set an integer major to
# enforce an absolute bar (e.g. 16).
CURRENT_DBR_MAJOR = None

# CO-03-03: set False if the environment does NOT do chargeback - the check then
# records open without scanning tags (confirm with the owners first).
CHARGEBACK_ENABLED = True
# Tag-key tokens that mark a billing record as cost-attributable (RLIKE, lowercased).
# Replace with the tag taxonomy your teams actually use.
CHARGEBACK_TAG_PATTERNS = ["cost", "team", "project", "owner", "business", "dept",
                           "department", "bu", "chargeback", "env"]

# ============================================================================
# Instance generation (CO-01-07) - per cloud, see CLOUD_PROVIDER
# ============================================================================

# Azure: the generation is the _vN suffix (Standard_D4ds_v5 -> 5; no suffix = 1).
# VM types at or above this version count as current generation.
AZURE_MIN_VM_VERSION = 5
# AWS / GCP: node types starting with one of these family prefixes count as current
# generation. None = built-in list for the cloud (checks/cloud.py).
MODERN_INSTANCE_FAMILIES = None

# ============================================================================
# Medallion layer naming - add the patterns your environment uses
# ============================================================================

# R-02-01 detects bronze/silver/gold layering by name (schema, then table, then
# catalog). These are RLIKE substrings (case-insensitive), including common
# abbreviations. Add the layer names your teams actually use.
MEDALLION_PATTERNS = {
    "bronze": ["bronze", "brz", "brnz", "raw", "landing", "ldg", "staging", "stg"],
    "silver": ["silver", "slv", "slvr", "cleansed", "conformed", "refined", "rfd", "curated"],
    "gold":   ["gold", "gld", "mart", "presentation", "prs", "serving", "semantic", "reporting", "rpt"],
}

# ============================================================================
# Service limits & quotas (OE-04-01) - documented Databricks limits
# ============================================================================

# OE-04-01 scores resource-quota UTILIZATION (current usage / documented limit) and is
# INVERSE (more consumed = worse):
#   fullest limit >= QUOTA_OPEN_AT                 -> open        (capacity action needed)
#   fullest limit >= QUOTA_COMPLETE_BELOW          -> in-progress (monitor, plan increases)
#   else                                           -> completed   (ample headroom)
# Scored on the FULLEST limit, not the average, so many idle limits cannot hide one
# that is close to its ceiling. Every limit is still listed in the rationale.
# Per-workspace limits are checked for EVERY workspace (from system tables / the audit
# log) and the fullest workspace is compared with the limit. Account limits (users +
# service principals, groups) come from the workspace account-SCIM endpoint.
# The rationale also reports an EFFICIENCY view (active vs registered secret scopes,
# Git folders, Apps, custom cluster policies) - informational, it does not change status.
# The documented limits live in checks/operational_excellence.DEFAULT_QUOTA_LIMITS.
# Many are per-account defaults that can be raised - if you have had any increased,
# put your EFFECTIVE limit here by ASSET KEY. Only the keys you list are overridden;
# every other key keeps its documented default. Valid keys and their defaults:
#   saved_jobs            12,000     Saved jobs (per workspace)
#   saved_pipelines       12,000     Saved Lakeflow pipelines (per workspace)
#   catalogs               1,000     Catalogs (per metastore)
#   schemas_per_catalog   10,000     Schemas in the fullest catalog
#   tables_per_schema     10,000     Tables in the fullest schema
#   tables_metastore   1,000,000     Tables (per metastore)
#   functions_per_schema  10,000     Functions in the fullest schema
#   volumes_per_schema    10,000     Volumes in the fullest schema
#   sql_warehouses         1,000     SQL warehouses (per workspace)
#   apps                     100     Databricks Apps (per workspace)
#   secret_scopes          1,000     Secret scopes (per workspace)
#   uc_registered_models   5,000     UC registered models (per metastore)
#   users_and_service_principals 10,000  Users + service principals (per account)
#   groups                 5,000     Groups (per account)
#
# Example - add several exceptions at once (jobs, apps, and warehouses raised):
#   QUOTA_LIMIT_OVERRIDES = {
#       "saved_jobs": 20000,
#       "apps": 250,
#       "sql_warehouses": 2000,
#   }
QUOTA_LIMIT_OVERRIDES = {}
QUOTA_COMPLETE_BELOW = 0.60   # fullest limit below this -> completed
QUOTA_OPEN_AT = 0.80          # fullest limit at/above this -> open

# ============================================================================
# Account-wide inventory & catalog visibility
# ============================================================================

# Per-workspace objects (secret scopes, Git folders/credentials, custom cluster
# policies, Apps) are rebuilt from system.access.audit create/delete events over this
# window. Objects created earlier and never touched since are not seen; each check
# reports a coverage comparison against the Workspace API for this workspace.
INVENTORY_WINDOW_DAYS = 365

# information_schema hides catalogs bound only to OTHER workspaces (no grant fixes
# that). When True, the run finds them (audit log + bindings API) and stamps a
# [VISIBILITY] caveat on catalog-scoped results. Bind them read-only to this
# workspace to include them.
DETECT_HIDDEN_CATALOGS = True


# ============================================================================
# Interoperability - certified partner tools (IU-01-03)
# ============================================================================

# client_application tokens treated as certified partner BI/ETL tools. None = use the
# built-in list (checks/interoperability.DEFAULT_PARTNER_TOOLS); set a list to match
# the toolchain your teams actually use.
PARTNER_TOOLS = None

# ============================================================================
# Scoring - defaults follow the WAF Assessment Tool convention
# ============================================================================

# Activity window for telemetry checks (job runs, queries, billing, lineage).
LOOKBACK_DAYS = 90

# Coverage at or above this fraction scores "completed"; below scores
# "in-progress". 0.80 = the 80% rule.
COMPLETE_AT = 0.80

# R-02-05: minimum upstream lineage hops an ML table must have to count as
# data-centric. 2 = ML should read a curated/silver layer, not raw; 1 = any upstream.
ML_MIN_UPSTREAM_HOPS = 2

# ============================================================================
# Optional
# ============================================================================

CREATE_CATALOG = False       # True to CREATE CATALOG IF NOT EXISTS
RESULTS_TABLE = "waf_assessment_results"
MATERIALIZE_LINEAGE = True   # cache system.access.*_lineage into the schema so
                             # lineage checks run fast; needs PERSIST_RESULTS = True
PERSIST_RESULTS = True       # False to run read-only with no results table
DEBUG = False                # True to print tracebacks from failing checks
RUN_ID = None                # None generates a timestamped id

## 2. Validation and shared context

This cell validates your settings, makes the repo importable, and builds the shared
`ctx` object the checks run against. It fails loudly if a required value is missing.

In [0]:
import os
import sys

# Make the repo root importable regardless of where this notebook lives.
_here = os.path.dirname(os.path.abspath("__file__" if "__file__" not in dir() else __file__))
try:
    _nb_dir = os.path.dirname(
        dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        .notebookPath().get()
    )
    _candidates = ["/Workspace" + _nb_dir, "/Workspace" + os.path.dirname(_nb_dir), _here]
except Exception:
    _candidates = [_here, os.getcwd()]

REPO_ROOT = None
for _c in _candidates:
    if _c and os.path.exists(os.path.join(_c, "waf_core.py")):
        REPO_ROOT = _c
        break
    _parent = os.path.dirname(_c) if _c else None
    if _parent and os.path.exists(os.path.join(_parent, "waf_core.py")):
        REPO_ROOT = _parent
        break

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate waf_core.py. Ensure this notebook is inside the "
        "waf_current_state_scripts repo (Git folder) and that waf_core.py sits at "
        "the repo root."
    )
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# --- validate ---------------------------------------------------------------
_missing = [n for n, v in [("CATALOG", CATALOG), ("SCHEMA", SCHEMA)] if not str(v).strip()]
if _missing and PERSIST_RESULTS:
    raise ValueError(
        f"Set {' and '.join(_missing)} in the cell above before running. "
        "(Or set PERSIST_RESULTS = False to run without a results table.)"
    )
if not 0 < COMPLETE_AT <= 1:
    raise ValueError(f"COMPLETE_AT must be between 0 and 1, got {COMPLETE_AT}")
if int(LOOKBACK_DAYS) < 1:
    raise ValueError(f"LOOKBACK_DAYS must be at least 1, got {LOOKBACK_DAYS}")
if str(CLOUD_PROVIDER).strip().lower() not in ("azure", "aws", "gcp"):
    raise ValueError(f'CLOUD_PROVIDER must be "azure", "aws" or "gcp", got {CLOUD_PROVIDER!r}')

import waf_core as wc
import waf_questions as wq

CFG = {
    "catalog": CATALOG.strip(),
    "schema": SCHEMA.strip(),
    "cloud_provider": str(CLOUD_PROVIDER).strip().lower(),
    "azure_min_vm_version": int(AZURE_MIN_VM_VERSION),
    "modern_instance_families": MODERN_INSTANCE_FAMILIES,
    "results_table": RESULTS_TABLE,
    "materialize_lineage": bool(MATERIALIZE_LINEAGE),
    "prod_catalog_patterns": PROD_CATALOG_PATTERNS,
    "prod_schema_exclude_patterns": PROD_SCHEMA_EXCLUDE_PATTERNS,
    "system_catalogs": SYSTEM_CATALOGS,
    "medallion_patterns": MEDALLION_PATTERNS,
    "lookback_days": int(LOOKBACK_DAYS),
    "complete_at": float(COMPLETE_AT),
    "ml_min_upstream_hops": int(ML_MIN_UPSTREAM_HOPS),
    "quota_limit_overrides": QUOTA_LIMIT_OVERRIDES,
    "quota_complete_below": float(QUOTA_COMPLETE_BELOW),
    "quota_open_at": float(QUOTA_OPEN_AT),
    "inventory_window_days": int(INVENTORY_WINDOW_DAYS),
    "detect_hidden_catalogs": bool(DETECT_HIDDEN_CATALOGS),
    "current_dbr_major": CURRENT_DBR_MAJOR,
    "chargeback_enabled": bool(CHARGEBACK_ENABLED),
    "chargeback_tag_patterns": CHARGEBACK_TAG_PATTERNS,
    "partner_tools": PARTNER_TOOLS,
    "create_catalog": bool(CREATE_CATALOG),
    "persist_results": bool(PERSIST_RESULTS),
    "debug": bool(DEBUG),
    "run_id": RUN_ID,
}

ctx = wc.Ctx(CFG, spark=spark)

print(f"Repo root      : {REPO_ROOT}")
print(f"Run ID         : {ctx.run_id}")
print(f"Cloud          : {ctx.cloud}")
print(f"Lookback       : {ctx.lookback_days} days")
print(f"Complete at    : {ctx.complete_at:.0%}")
if PERSIST_RESULTS:
    print(f"Results table  : {wc.results_table_fqn(CFG)}")
else:
    print("Results table  : (disabled)")

## 3. Resolve assessment scope

Resolving the scope now surfaces a mis-set `PROD_CATALOG_PATTERNS` immediately, rather
than as a puzzling wall of `open` results later.

In [0]:
scope = ctx.scope

print(f"Scope mode : {scope.mode}")
print(f"Catalogs   : {len(scope.catalogs)}")
for c in scope.catalogs[:25]:
    print(f"   - {c}")
if len(scope.catalogs) > 25:
    print(f"   ... and {len(scope.catalogs) - 25} more")

if scope.mode == "all":
    print(
        "\nWARNING: no catalog matched PROD_CATALOG_PATTERNS "
        f"({PROD_CATALOG_PATTERNS}).\n"
        "Falling back to ALL non-system catalogs. Every affected rationale will say so.\n"
        "If these results should reflect production only, update "
        "PROD_CATALOG_PATTERNS above and re-run."
    )
elif not scope.catalogs:
    print("\nWARNING: scope is empty - no catalogs to assess.")
else:
    print("\nScope resolved to production catalogs.")

## 4. Verify system table access

Checks degrade to `open` with an explanatory rationale when a system schema is
unreadable, so the assessment still runs — but a lot of `open` results usually means
missing grants rather than missing controls. Confirm here before drawing conclusions.

In [0]:
REQUIRED_TABLES = [
    ("system.information_schema.tables", "catalog metadata (most pillars)"),
    ("system.information_schema.columns", "column metadata / documentation"),
    ("system.access.audit", "audit logging (governance, ops)"),
    ("system.access.table_lineage", "lineage (governance, interoperability)"),
    ("system.billing.usage", "cost and serverless adoption"),
    ("system.compute.clusters", "compute configuration"),
    ("system.compute.warehouses", "SQL warehouse configuration"),
    ("system.query.history", "query performance and monitoring"),
    ("system.lakeflow.jobs", "job orchestration and reliability"),
    ("system.lakeflow.pipelines", "declarative pipelines"),
    ("system.storage.table_metrics_history", "file sizes, predictive optimization"),
    ("system.mlflow.runs_latest", "ML experiment tracking"),
    ("system.serving.served_entities", "model serving"),
]

available, unavailable = [], []
for fqn, why in REQUIRED_TABLES:
    (available if ctx.has_table(fqn) else unavailable).append((fqn, why))

print(f"Readable: {len(available)}/{len(REQUIRED_TABLES)} system tables\n")
for fqn, why in available:
    print(f"  [x] {fqn:<50} {why}")
if unavailable:
    print()
    for fqn, why in unavailable:
        print(f"  [ ] {fqn:<50} {why}")
    print(
        "\nUnreadable tables produce 'open' results with an explanatory rationale.\n"
        "If that is unexpected, grant SELECT on the system schemas to the principal "
        "running this assessment."
    )

## 5. Helper used by the pillar notebooks

`run_pillar` is the one entry point each pillar notebook calls. Keeping it here means the
pillar notebooks stay thin and consistent.

In [0]:
import importlib


def run_pillar(module_name: str, pillar_id: str, persist: bool = None):
    """Run one pillar's checks, print its report, optionally persist, return results.

    Args:
        module_name: module under ``checks/`` (e.g. "checks.governance").
        pillar_id: WAF pillar id (e.g. "data-ai-governance").
        persist: override ``PERSIST_RESULTS`` for this run.
    """
    mod = importlib.import_module(module_name)
    importlib.reload(mod)  # pick up edits without detaching the notebook

    pillar_name = wq.PILLAR_NAMES[pillar_id]
    meta = wq.pillar_meta(pillar_id)

    print(f"Assessing {pillar_name}: {len(mod.CHECKS)} question(s)\n")
    results = ctx.run_checks(pillar_name, meta, mod.CHECKS)

    # Fail loudly if a pillar's CHECKS list drifts from the published question bank.
    expected, got = set(meta), {r.qid for r in results}
    if expected - got:
        raise AssertionError(f"{pillar_name}: no check defined for {sorted(expected - got)}")
    if got - expected:
        raise AssertionError(f"{pillar_name}: unknown question ids {sorted(got - expected)}")

    print()
    print(wc.render_pillar_report(pillar_name, results))

    should_persist = CFG["persist_results"] if persist is None else persist
    if should_persist:
        wc.persist_results(ctx, results)
    return results


print("Configuration loaded. run_pillar() is ready.")

## Materialize lineage

Builds a lineage snapshot and its transitive closure once, up front, in the configured
schema (`waf_lineage_edges`, `waf_lineage_closure`, `waf_lineage_column_edges`) using the
iterative BFS pattern from [databricks_lineage](https://github.com/kailynklaassen/databricks_lineage). The
lineage-heavy checks (`DG-01-03`, `DG-02-01`, `IU-04-03`, `R-02-05`) then read these small
local tables instead of re-scanning `system.access.*_lineage`. Controlled by
`MATERIALIZE_LINEAGE`; runs only when `PERSIST_RESULTS` is on, and falls back to the system
tables if skipped.

In [0]:
_lineage = wc.build_lineage_tables(ctx)
if _lineage.get("skipped"):
    print(f"Lineage snapshot skipped ({_lineage['skipped']}); checks read system tables.")
elif _lineage.get("error"):
    print(f"Lineage snapshot NOT built ({_lineage['error']}); checks read system tables.")
else:
    print(
        f"Lineage materialized in {CFG['catalog']}.{CFG['schema']}: "
        f"{_lineage.get('edges', 0):,} edge(s), "
        f"{_lineage.get('closure', 0):,} closure pair(s) over {_lineage.get('hops', 0)} hop(s), "
        f"{_lineage.get('column_edges', 0):,} column edge(s)."
    )